<a href="https://colab.research.google.com/github/ancestor9/mathematics-for-machine-learning/blob/main/pytorch/09_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 학습 구성 (핵심 수치)

- 전체 샘플 수: 178개
- Batch Size: 3 (한 번에 3개씩 묶어서 학습)
- Iterations (Step): $178 \div 3 = 59.33... \rightarrow$ 60번
>> 즉, 1 Epoch(전체 데이터를 한 바퀴 도는 것) 동안 모델은 총 60번의 파라미터 업데이트를 수행

### 2. 학습 루프의 동작 과정
- 1번 Epoch 동안 DataLoader는 데이터를 다음과 같이 공급하며 학습을 진행

- 1. 데이터 쪼개기: 전체 178개 데이터를 3개씩 묶어 총 60개의 덩어리(Batch)를 생성
- 2.반복 업데이트:

> - Step 1: 1~3번 데이터 투입 $\rightarrow$ 손실 계산 $\rightarrow$ 1번째 가중치 수정

> - Step 2: 4~6번 데이터 투입 $\rightarrow$ 손실 계산 $\rightarrow$ 2번째 가중치 수정...

> - (중략) ...

> - Step 60: 마지막 남은 데이터를 투입 $\rightarrow$ 손실 계산 $\rightarrow$ 60번째 가중치 수정

### 3. 핵심 요약
- 업데이트 횟수: 배치 사이즈가 4일 때(45번)보다 3일 때(60번) 더 자주 모델을 수정
- 학습 특징: 배치가 작아질수록 모델을 더 세밀하고 자주 고치게 되지만, 한 번에 처리하는 데이터 양이 적어 학습 속도는 조금 더 걸릴 수 있음

결론적으로 배치 사이즈가 3이라는 것은 "30명의 학생이 있는 반에서 3명씩 상담을 진행하여, 총 10번의 상담을 통해 반 전체 상황을 파악(1 Epoch)하는 것"과 같습니다. 178개의 데이터라면 이 상담(업데이트)을 60번 하는 셈

### 효율적인 학습을 위한 pytorch 데이터 처리 작동방식
- 데이터셋 분할: 전체 데이터를 한 번에 학습시키면 메모리가 부족할 수 있으므로, 데이터를 작은 묶음인 Batch(배치) 단위로 나누어 학습

- 용어 정리:

>> Epoch: 전체 데이터를 한 번 다 본 상태.

>> Batch Size: 한 번의 업데이트에 사용하는 샘플 개수.

>> Iteration: 1 Epoch을 완료하기 위한 배치 반복 횟수.

- Dataset & DataLoader: PyTorch의 이 도구들은 데이터를 섞고(Shuffle), 병렬로 로드하며, 배치 크기에 맞춰 자동으로 공급해 주는 역할

In [1]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math

# gradient computation etc. not efficient for whole data set
# -> divide dataset into small batches

'''
# training loop
for epoch in range(num_epochs):
    # loop over all batches
    for i in range(total_batches):
        batch_x, batch_y = ...
'''

# epoch = one forward and backward pass of ALL training samples
# batch_size = number of training samples used in one forward/backward pass
# number of iterations = number of passes, each pass (forward+backward) using [batch_size] number of sampes
# e.g : 100 samples, batch_size=20 -> 100/20=5 iterations for 1 epoch

# --> DataLoader can do the batch computation for us

# Implement a custom Dataset:
# inherit Dataset
# implement __init__ , __getitem__ , and __len__

class WineDataset(Dataset):

    def __init__(self):
        # Initialize data, download, etc.
        # read with numpy or pandas
        xy = np.loadtxt('wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = xy.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(xy[:, 1:]) # size [n_samples, n_features]
        self.y_data = torch.from_numpy(xy[:, [0]]) # size [n_samples, 1]

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples


# create dataset
dataset = WineDataset()

# get first sample and unpack
first_data = dataset[0]
features, labels = first_data
print(features, labels)

# Load whole dataset with DataLoader
# shuffle: shuffle data, good for training
# num_workers: faster loading with multiple subprocesses
# !!! IF YOU GET AN ERROR DURING LOADING, SET num_workers TO 0 !!!
train_loader = DataLoader(dataset=dataset,
                          batch_size=4,
                          shuffle=True,
                          num_workers=2)

# convert to an iterator and look at one random sample
dataiter = iter(train_loader)
data = next(dataiter)
features, labels = data
print(features, labels)

# Dummy Training loop
num_epochs = 2
total_samples = len(dataset)
n_iterations = math.ceil(total_samples/4)
print(total_samples, n_iterations)
for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(train_loader):

        # here: 178 samples, batch_size = 4, n_iters=178/4=44.5 -> 45 iterations
        # Run your training process
        if (i+1) % 5 == 0:
            print(f'Epoch: {epoch+1}/{num_epochs}, Step {i+1}/{n_iterations}| Inputs {inputs.shape} | Labels {labels.shape}')

# some famous datasets are available in torchvision.datasets
# e.g. MNIST, Fashion-MNIST, CIFAR10, COCO

train_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=True,
                                           transform=torchvision.transforms.ToTensor(),
                                           download=True)

train_loader = DataLoader(dataset=train_dataset,
                                           batch_size=3,
                                           shuffle=True)

# look at one random sample
dataiter = iter(train_loader)
data = next(dataiter)
inputs, targets = data
print(inputs.shape, targets.shape)

tensor([1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
        3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
        1.0650e+03]) tensor([1.])
tensor([[1.3170e+01, 5.1900e+00, 2.3200e+00, 2.2000e+01, 9.3000e+01, 1.7400e+00,
         6.3000e-01, 6.1000e-01, 1.5500e+00, 7.9000e+00, 6.0000e-01, 1.4800e+00,
         7.2500e+02],
        [1.2600e+01, 1.3400e+00, 1.9000e+00, 1.8500e+01, 8.8000e+01, 1.4500e+00,
         1.3600e+00, 2.9000e-01, 1.3500e+00, 2.4500e+00, 1.0400e+00, 2.7700e+00,
         5.6200e+02],
        [1.2840e+01, 2.9600e+00, 2.6100e+00, 2.4000e+01, 1.0100e+02, 2.3200e+00,
         6.0000e-01, 5.3000e-01, 8.1000e-01, 4.9200e+00, 8.9000e-01, 2.1500e+00,
         5.9000e+02],
        [1.4380e+01, 3.5900e+00, 2.2800e+00, 1.6000e+01, 1.0200e+02, 3.2500e+00,
         3.1700e+00, 2.7000e-01, 2.1900e+00, 4.9000e+00, 1.0400e+00, 3.4400e+00,
         1.0650e+03]]) tensor([[3.],
        [2.],
        [3.],
        [1.]])
178 45
Epoch

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.83MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.44MB/s]

torch.Size([3, 1, 28, 28]) torch.Size([3])


In [3]:
dataset

In [4]:
print(f"Number of samples in dataset: {len(dataset)}")
print(f"Shape of features (x_data): {dataset.x_data.shape}")
print(f"Shape of labels (y_data): {dataset.y_data.shape}")

Number of samples in dataset: 178
Shape of features (x_data): torch.Size([178, 13])
Shape of labels (y_data): torch.Size([178, 1])


### 1. inputs.shape: torch.Size([3, 1, 28, 28])

- 이미지 데이터를 담고 있는 텐서의 차원(Batch_size, Channels, Height, Width) 형식

>> 3 (Batch Size): 한 번에 불러온 이미지의 개수로 DataLoader에서 batch_size=3으로 설정했기 때문에 3개의 이미지가 묶여 있음

>> 1 (Channels): 이미지의 색상 채널 수로 MNIST는 흑백 이미지이므로 채널이 1개 (컬러 RGB였다면 3).

>> 28 (Height): 이미지의 세로 픽셀 수

>> 28 (Width): 이미지의 가로 픽셀 수

- "가로 28, 세로 28 픽셀 크기의 흑백 이미지 3개가 하나의 묶음(Batch)으로 되어 있다"는 뜻

### 2. targets.shape: torch.Size([3])
- 각 이미지에 대한 정답(라벨)을 담고 있는 텐서의 차원

>> 3: 입력받은 이미지 3개에 대응하는 3개의 숫자 라벨

>>> 각 값은 0부터 9 사이의 정수(예: tensor([5, 0, 4]))로, 해당 이미지가 숫자 몇인지 나타냄

In [2]:
print(inputs.shape, targets.shape)

torch.Size([3, 1, 28, 28]) torch.Size([3])


- 입력변수 : [3, 1, 28, 28]: [배치 크기, 채널(흑백/컬러), 가로, 세로] 즉, 28x28 pixel 크기의 흑백 이미지 3개가 한 묶음(batch)이라는 뜻(1은 channel이 1개, 즉 흑백)

- 출력변수 : [3]: 그 이미지 3개에 대응하는 각각의 정답(라벨) 값으로 벡터(정답이 3개)